# **Entity Extraction :**

There are several models known for Named Entity Recognition tasks. The choice of the model depends on the dataset. This notebook explores how LayoutLM performs, given the token level extractions for each invoice image is provided along with the OCR transcript.

LayoutLM integrates bounding box information directly with the token embeddings, providing a better framework for handling invoices where the structure and positioning of entities are critical. For this task, the [LayoutLM base-uncased model](https://huggingface.co/microsoft/layoutlm-base-uncased) is used which doesn't require the actual images for training, further reducing the computational requirements.

Although the tokens are generated the same way as any other language model (like BERT), the spatial information is treated as an additional feature for each token and it's encoded as embeddings in the tokens. Even during inference, the model performs well only with the OCR extracted text and spatial arrangement, without needing the pixel level information.

In [ ]:
# Imports
import torch
import os
import pandas as pd
import numpy as np
from transformers import LayoutLMTokenizer
from PIL import Image
import evaluate

In [ ]:
# Check if GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("CUDA device is available and being used.")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("MPS device is available and being used.")
else:
    device = torch.device('cpu')
    print("MPS device is not available, using CPU instead.")

List all the entities to be extracted and convert the labels to IDs

In [ ]:
entity_labels = [
    'OTHER',
    'employerName',
    'employerAddressStreet_name',
    'employerAddressCity',
    'employerAddressState',
    'employerAddressZip',
    'einEmployerIdentificationNumber',
    'employeeName',
    'ssnOfEmployee',
    'box1WagesTipsAndOtherCompensations',
    'box2FederalIncomeTaxWithheld',
    'box3SocialSecurityWages',
    'box4SocialSecurityTaxWithheld',
    'box16StateWagesTips',
    'box17StateIncomeTax',
    'taxYear',
]

label2id = {label: idx for idx, label in enumerate(entity_labels)}
id2label = {idx: label for label, idx in label2id.items()}

In [ ]:
# Initialize LayoutLM tokenizer
tokenizer = LayoutLMTokenizer.from_pretrained("microsoft/layoutlm-base-uncased", padding='max_length', truncation=True, max_length=128)

The **process_document** function reads both the invoice image and it's corresponding TSV file containing the word-level token extractions. The image is not required for training but to standardize every image to a fixed size, the size of image is required.

Task of the function :

1.   Retrive the size of image
2.   Scale the image to a fixed size 1000x1000 (Since LayoutLM uses relative spatial encoding 0-1000 grid)
3.   Convert the coordinates as per scaled dimentions
4.   Convert the labels to IDs
5.   Tokenize the OCR words, modified bounding boxes and label IDs
6.   Return the information structured in a LayoutLM specific format

If both TSV and Image files exist, pass the TSV file into the process document function and append the returned dictionary of information




In [ ]:
# Function to process a single document of TRAINING
def process_document(tsv_file, image_file, label2id=None):
    # Read the TSV file into a pandas dataframe
    df = pd.read_csv(tsv_file, header=None)

    # Read the image to get its dimensions (optional for LayoutLM v1)
    image = Image.open(image_file)
    width, height = image.size

    x_scale = 1000 / width
    y_scale = 1000 / height

    # Add a header (column names)
    if len(df.columns) == 8:
        df.columns = ['start_index', 'end_index', 'x1', 'y1', 'x2', 'y2', 'text', 'label']
    else:
        df.columns = ['start_index', 'end_index', 'x1', 'y1', 'x2', 'y2', 'text']
    
    # Drop the 'start_index' and 'end_index' columns
    df = df.drop(columns=['start_index', 'end_index'])

    # Scaling the bounding boxes to fit the image of size 1000x1000
    df[['x1', 'x2']] = df[['x1', 'x2']] * x_scale
    df[['y1', 'y2']] = df[['y1', 'y2']] * y_scale

    # If the scale values were in floating points, the coordinate values will also become floating when multiplied
    # Since LayoutLM expects only integers as the bounding boxes values, round off the figure and cast it as 'int'
    df[['x1', 'x2', 'y1', 'y2']] = df[['x1', 'x2', 'y1', 'y2']].round().astype(int)

    # Clip the values to enfore 0-1000 range
    # Even after standardising, due to noisy data some values may overflow
    df[['x1', 'x2', 'y1', 'y2']] = df[['x1', 'x2', 'y1', 'y2']].clip(0, 1000)

    # Convert the 'text' column to string type for tokenizer to work without error
    df['text'] = df['text'].astype(str)

    # Drop duplicates
    df = df.drop_duplicates(subset=['x1', 'y1', 'x2', 'y2', 'text'], keep='first')

    words = df['text'].tolist()
    bboxes = df[['x1', 'y1', 'x2', 'y2']].values.tolist()

    if label2id is not None and 'label' in df.columns:
        raw_labels = df['label'].tolist()
        # Converting the labels into label_ids
        labels = [label2id[label] for label in raw_labels]
    else:
        labels = None

    tokenized_words = []
    token_bboxes = []
    token_labels = [] if labels is not None else None

    # Process each word in the document
    for i, (word, bbox) in enumerate(zip(words, bboxes)):
        word = str(word)
        # Tokenize the word
        tokenized_word = tokenizer.tokenize(word)
        tokenized_words.extend(tokenized_word)

        # Add the same bounding box for each subword
        token_bboxes.extend([bbox] * len(tokenized_word))

        # Add the same label for each subword
        if labels is not None:
            label = labels[i]
            token_labels.extend([label] * len(tokenized_word))

    # Convert tokenized words to input IDs
    input_ids = tokenizer.convert_tokens_to_ids(tokenized_words)

    # Create attention masks (1 for tokens, 0 for padding, if any)
    attention_mask = [1] * len(input_ids)

    # Format the document into the required structure
    document = {
        "id": os.path.basename(tsv_file).split('.')[0],  # Use the file name as the document ID
        "input_ids": torch.tensor(input_ids),
        "bbox": torch.tensor(token_bboxes),
        "attention_mask": torch.tensor(attention_mask)
    }

    if token_labels is not None:
        document["labels"] = torch.tensor(token_labels)

    return document


In [ ]:
# Function to load the documents
def load_documents(tsv_dir, image_dir, label2id):
    documents =[]

    for tsv_file in os.listdir(tsv_dir):
        if tsv_file.endswith('.tsv'):
            tsv_path = os.path.join(tsv_dir, tsv_file)
            image_name = tsv_file.replace('.tsv', '.jpg')  # for images that are .jpg, adjust if necessary
            image_path = os.path.join(image_dir, image_name)

            if os.path.exists(image_path):
                document = process_document(tsv_path, image_path, label2id)
                documents.append(document)
            else:
                print(f"Image for {tsv_file} not found.")

    return documents

In [ ]:
# Get the root path of the project directory, which is one level up from the current working directory 
BASE_DIR = os.path.dirname(os.getcwd())

In [ ]:
# Dataset paths for directories containing the TSV files and corresponding images of train data
TRAIN_TSV_DIR = os.path.join(BASE_DIR, "dataset", "train", "boxes_transcripts_labels")
TRAIN_IMAGE_DIR = os.path.join(BASE_DIR, "dataset", "train", "images")

# Dataset paths for directories containing the TSV files and corresponding images of test data
TEST_TSV_DIR = os.path.join(BASE_DIR, "dataset", "test", "boxes_transcripts")
TEST_IMAGE_DIR = os.path.join(BASE_DIR, "dataset", "test", "images")

In [ ]:
# Process all documents in the tsv directory of train data
all_documents = load_documents(TRAIN_TSV_DIR, TRAIN_IMAGE_DIR, label2id=label2id)

In [ ]:
# Process all documents in the tsv directory of test data
all_documents_test = load_documents(TEST_TSV_DIR, TEST_IMAGE_DIR, label2id=None)

Since each invoice image has variable number of entries (i.e words extracted), for every processed file (i.e for element of the list all_documents) pad all the keys of the dictionary with zeros to maximum length so that all the information of each file is of same length

## Chunking

Since the LayoutLM model can handle the tokens of length 512, split each file information into sizes of 512. The last chunk is padded appropriately to match the length of 512.

In [ ]:
# function to split the data as per model requirements

def split_data_into_chunks(full_dataset, chunk_size=512):
    new_dataset = []

    for item in full_dataset:
        input_ids = item['input_ids']
        bbox = item['bbox']
        attention_mask = item['attention_mask']
        labels = item['labels']  if 'labels' in item else None
        id_value = item['id']  # Keep the ID the same

        total_len = len(input_ids)

        for chunk_idx, i in enumerate(range(0, total_len, chunk_size)):

            chunk_input_ids = input_ids[i:i+chunk_size]
            chunk_bbox = bbox[i:i+chunk_size]
            chunk_attention_mask = attention_mask[i:i+chunk_size]

            if labels is not None:
                chunk_labels = labels[i:i+chunk_size]
            
            # Padding
            pad_len = chunk_size - len(chunk_input_ids)

            if pad_len > 0:
                chunk_input_ids = torch.cat([chunk_input_ids, torch.zeros(pad_len, dtype=torch.long)])
                chunk_bbox = torch.cat([chunk_bbox, torch.zeros((pad_len,4), dtype=torch.long)])
                chunk_attention_mask = torch.cat([chunk_attention_mask, torch.zeros(pad_len, dtype=torch.long)])

                if labels is not None:
                    chunk_labels = torch.cat([chunk_labels, torch.full((pad_len,), -100, dtype=torch.long)])

            new_item = {
                'id': id_value,
                'chunk_idx': chunk_idx,
                'input_ids': chunk_input_ids,
                'bbox': chunk_bbox,
                'attention_mask': chunk_attention_mask,
            }
            if labels is not None:
                new_item['labels'] = chunk_labels

            new_dataset.append(new_item)

    return new_dataset

In [ ]:
from sklearn.model_selection import train_test_split

# Split list into 80% training and 20% validation
train_data, val_data = train_test_split(all_documents, test_size=0.2, random_state=42)

In [ ]:
# Split the train, validation & test data into chunks of 512 (LayoutLM model requirement)
chunk_size = 512

train_data_split = split_data_into_chunks(train_data, chunk_size=chunk_size)
val_data_split = split_data_into_chunks(val_data, chunk_size=chunk_size)
test_data_split = split_data_into_chunks(all_documents_test, chunk_size=chunk_size)

Prepare the dataset to feed it into the network for training

In [ ]:
from torch.utils.data import Dataset

# Wrap the dataset as per the data loader reqirements (Since the training loops expect both len(dataset) & dataset[i])
class LayoutLMDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        output =  {
            'input_ids': item['input_ids'],
            'bbox': item['bbox'],
            'attention_mask': item['attention_mask'],
        }

        if 'labels' in item:
            output['labels'] = item['labels']

        return output

In [ ]:
# Initialize the dataset and model
train_dataset = LayoutLMDataset(train_data_split)
val_dataset = LayoutLMDataset(val_data_split)
test_dataset = LayoutLMDataset(test_data_split)

Import the pretrained LayoutLM model from HuggingFace

In [ ]:
from transformers import LayoutLMForTokenClassification

In [ ]:
# Load the pre-trained LayoutLM model for token classification
model = LayoutLMForTokenClassification.from_pretrained(
    "microsoft/layoutlm-base-uncased",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
from transformers import DataCollatorForTokenClassification

In [ ]:
# Data collator ensures correct formatting for token classification tasks
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True) # data collator to handle padding

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

Define a function to keep track of the evaluation metrics during training

In [ ]:
# Load the metric
metric = evaluate.load("seqeval")  # For sequence labeling tasks

# Define a function to compute metrics
def compute_metrics(p):
    predictions, labels = p
    # Convert the predicted label IDs to actual labels
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (usually -100 in the labels)
    true_labels = [[entity_labels[l] for l in label if l != -100] for label in labels]
    predicted_labels = [[entity_labels[pred] for pred, lab in zip(prediction, label) if lab != -100]
                        for prediction, label in zip(predictions, labels)]

    # Compute the metrics using seqeval
    results = metric.compute(predictions=predicted_labels, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

**Setup the model for training :**

Since LayoutLM is already pretrained with huge data, few epochs are enough to finetune it with a dataset of modest size(<10,000).

Early stopping will ensure the model stops if the model doesn't show improvement in validation performance for 3 consecutive epochs.

TensorBoard is used to view the real-time model performance.

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=20,
    eval_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",  # Save model after every epoch
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to=["tensorboard"]          # Report to TensorBoard
)

# Define Trainer with model and training dataset
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# Train the model
trainer.train()

In [ ]:
# Path to save the model
MODEL_PATH = os.path.join(BASE_DIR, "models", "layoutlm_invoice_model")

MODEL_SAVE_PATH = os.path.join(BASE_DIR, "models", "layoutlm_invoice_model")

# Save the trained model
trainer.save_model("MODEL_SAVE_PATH")

In [ ]:
# Inference using trainer.predict()
predictions, _, _ = trainer.predict(test_dataset)

In [ ]:
# Convert predictions from numpy to torch tensor
predictions_tensor = torch.tensor(predictions)

# Step 1: Convert logits to predicted label IDs (take the argmax over num_labels)
predicted_label_ids = torch.argmax(predictions_tensor, dim=-1)

# Step 2: Map predicted label IDs to actual entity labels
predicted_entities = [[id2label[label_id.item()] for label_id in example] for example in predicted_label_ids]

Since each file was chunked to math the 512 length requirements, recombine the predictions. The tokenizer will break the words into subwords and sends them to train. However, the same bounding boxes are assigned to all it's subwords. Merge the subwords based on same bounding box values. Drop the duplicates to get the predictions for the original document.

In [ ]:
from collections import defaultdict

grouped_preds = defaultdict(list)

for item, pred in zip(test_data_split, predicted_entities):
    grouped_preds[item['id']].append({
            "chunk_idx": item['chunk_idx'],
            "predictions": pred,
            "attention_mask": item['attention_mask'],
            "bbox": item['bbox']
        })

final_output = []

for doc_id in grouped_preds:

    chunks = sorted(grouped_preds[doc_id], key=lambda x: x['chunk_idx'])

    combined_preds = []
    combined_masks = []
    combined_bbox = []

    for chunk in chunks:
        combined_preds.extend(chunk["predictions"])
        combined_masks.extend(chunk["attention_mask"].tolist())
        combined_bbox.extend(chunk["bbox"].tolist())

    # Strip the padded zeros in the merged documents using the attention mask
    cleaned_preds = [prediction for prediction, mask in zip(combined_preds, combined_masks) if mask==1]
    cleaned_bbox = [box for box, mask in zip(combined_bbox, combined_masks) if mask==1]

    # Detokenize - tokenization splits the words into subwords
    # Dropping the duplicates using the bbox will match the length of the original list of words
    df = pd.DataFrame({"pred": cleaned_preds, "bbox": cleaned_bbox})
    df = df.drop_duplicates(subset=["bbox"])

    # Format the final output by associating the predictions with its corresponding file name
    final_preds = df["pred"].tolist()

    final_output.append({"file_name": doc_id, "predictions": final_preds})

In [ ]:
print("\nFinal Output successfully computed")

print("\nSample Predictions:\n")
    
# Display output only of the first two for the sample
# Use print(output) to view the whole
print(final_output[:2])

# OPTIONAL UPGRADE : To better view the output 
import json
print(json.dumps(final_output[:2], indent=2))

In [ ]:
pred_map = {item["file_name"]: item["predictions"] for item in final_output}

for tsv_file in os.listdir(TEST_TSV_DIR):
    doc_id = tsv_file.split('.')[0]

    tsv_path = os.path.join(TEST_TSV_DIR, tsv_file)
    df = pd.read_csv(tsv_path, header=None)

    preds = pred_map.get(doc_id)

    assert len(df) == len(preds), f"Length mismatch in {doc_id}"